<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!

[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:

1. Add your solution to the cell with `## YOUR SOLUTION HERE ##`.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

#### Setup Cell
Run the setup cell to load the LangChain components provided in the setup file `langchain_setup.py`.
 
This setup file initializes everything required for a stateful conversation, including:
- `conversation_chain`: Executes the conversational pipeline by combining the prompt, model, and memory handling. It retrieves prior messages, sends the full request to the model, and stores responses back into memory.
- `get_session_history(session_id)`: Retrieves or creates `InMemoryChatMessageHistory` for a given session ID.
- `store`: Universal memory store (dictionary) that holds chat histories for all sessions, serving as a centralized repository where each session (identified by `session_id`) maintains its own message history, enabling multiple concurrent conversations.
 
The setup file also configures the underlying components used by the chain:
- `llm`: An OpenAI `GPT-3.5-turbo` model instance with temperature set to 0.5.
- `prompt`: A prompt template that defines the assistant’s role, includes conversation history, and accepts user input.
 
Because these components are already wired together in the setup file, you can focus on observing how conversation history affects request size rather than rebuilding the chain manually.
 
The `support_turns` list contains predefined user messages simulating a technical support conversation about a laptop startup issue.

In [3]:
# Local application imports
%run "./Implementing Your First Memory.ipynb"

# Define the support conversation turns
support_turns = [
    "My laptop won't start.",
    "I tried charging it overnight.",
    "The power light comes on but screen is black.",
    "I tried holding power button for 30 seconds.",
    "Yes, I removed all external devices.",
    "The laptop is 2 years old, a Dell XPS 15.",
    "No recent updates that I know of.",
    "I can hear the fan running.",
]

ModuleNotFoundError: No module named 'nbformat'

**Note:** Be sure to run the Setup cell above before beginning with the checkpoints! 

#### Checkpoint 1: Running a Multi-Turn Support Conversation With Buffer Memory

Run through a complete support conversation to see buffer memory in action as it retains every message exchanged.

1. Define a session configuration dictionary named `buffer_config` that specifies the `"session_id"` as `"support_buffer"` to create a unique session identifier that isolates the memory storage for this conversation.

2. A loop structure with `print()` statements is provided to run through the conversation and display each turn. 

3. Inside the loop, invoke the `conversation_chain` with the current `turn` as `"input"` and the session configuration, storing the result in the `response` variable so each message is sent to the chain while maintaining session context for conversation continuity.

**Note:** Don't forget to run the cell and save the notebook before selecting "Test Work". Open the "Jupyter Help" toggle at the top of the notebook for more details.

In [3]:
# Run buffer conversation
buffer_config = {"configurable": {"session_id": "support_buffer"}}

print("Running conversation with buffer memory...\n")
for i, turn in enumerate(support_turns, 1):
    response = conversation_chain.invoke({"input": turn}, config=buffer_config)
    print(f"Turn {i}: {turn}")

Running conversation with buffer memory...

Turn 1: My laptop won't start.
Turn 2: I tried charging it overnight.
Turn 3: The power light comes on but screen is black.
Turn 4: I tried holding power button for 30 seconds.
Turn 5: Yes, I removed all external devices.
Turn 6: The laptop is 2 years old, a Dell XPS 15.
Turn 7: No recent updates that I know of.
Turn 8: I can hear the fan running.


#### Checkpoint 2: Analyzing Buffer Memory Usage

Take a closer look at what's stored in buffer memory and calculate how much it grows with each conversation turn.

1. Retrieve the chat history for the `"support_buffer"` session and store the messages in a variable named `buffer_messages` to access the complete conversation history stored in memory for analysis.

2. Join all message contents into a single string using spaces and store it in a variable named `buffer_content` to create a single text representation that enables token estimation.

3. Estimate the token count by multiplying the number of words in `buffer_content` by `1.3` to get an approximate token count, since tokens don't map one-to-one with words. Store the result in a variable named `buffer_tokens`.

The `print()` statements are provided to display the buffer memory analysis results.

**Note:** Don't forget to run the cell and save the notebook before selecting "Test Work". Open the "Jupyter Help" toggle at the top of the notebook for more details.

In [4]:
# Analyze buffer memory usage
buffer_messages = get_session_history("support_buffer").messages
buffer_content = " ".join([m.content for m in buffer_messages])
buffer_tokens = len(buffer_content.split()) * 1.3  # rough estimate

print("=== BUFFER MEMORY ANALYSIS ===\n")
print(f"Total messages stored: {len(buffer_messages)}")
print(f"  User messages: {len([m for m in buffer_messages if m.type == 'human'])}")
print(f"  Assistant messages: {len([m for m in buffer_messages if m.type == 'ai'])}")
print(f"\nEstimated tokens: {buffer_tokens:.0f}")
print(f"Tokens per turn: {buffer_tokens / 8:.0f}")

print(f"\nObservation: Memory grows by ~2 messages per turn")

=== BUFFER MEMORY ANALYSIS ===

Total messages stored: 16
  User messages: 8
  Assistant messages: 8

Estimated tokens: 8683
Tokens per turn: 1085

Observation: Memory grows by ~2 messages per turn


#### Understanding How Tokens Accumulate

- **System message:** A set of instructions given to the model by the application, such as defining the model’s role, behavior, or response style.
- **User message:** The input sent by the user, representing a question, request, or instruction the model should respond to.
- **Assistant message:** The response generated by the model based on the system instructions and the user’s message.

**What gets sent to the LLM each turn:**

Turn 1:
- System message
- User message 1
- → LLM responds with Assistant message 1

Turn 2:
- System message
- User message 1
- Assistant message 1
- User message 2
- → LLM responds with Assistant message 2

Turn 3:
- System message
- User message 1
- Assistant message 1
- User message 2
- Assistant message 2
- User message 3
- → LLM responds with Assistant message 3

**The pattern:** Every turn sends all previous messages as context.

**Why 4000 tokens matters:** GPT-3.5-turbo has a 4,096 to 16,385 tokens limit in total context length. If your prompt exceeds this, the API call fails.

Now let's calculate when buffer memory will hit this token limit. 

We'll project token usage for conversations of different lengths (2, 5, 10, 15, 20, 25, and 30 turns) by multiplying the average tokens per turn by the number of turns. Each projection shows whether we're safe, approaching the limit, or already exceeding it. 

This reveals exactly when buffer memory will fail due to its linear growth pattern with no message removal.

In [5]:
# Project token growth
tokens_per_turn = buffer_tokens / 8
token_limit = 16385

print("=== TOKEN GROWTH PROJECTION ===\n")
for turns in [2, 5, 10, 15, 20, 25, 30]:
    projected = tokens_per_turn * turns
    if projected > token_limit:
        status = "❌ EXCEEDS LIMIT"
    elif projected > 3500:
        status = "⚠️ APPROACHING LIMIT"
    else:
        status = "✓"
    print(f"{turns:>3} turns: {projected:>6.0f} tokens  {status}")

failure_turn = token_limit / tokens_per_turn

print(f"\nBuffer memory will fail around turn {failure_turn:.0f}")
print(f"Reason: Linear token growth with no message removal")

=== TOKEN GROWTH PROJECTION ===

  2 turns:   2171 tokens  ✓
  5 turns:   5427 tokens  ⚠️ APPROACHING LIMIT
 10 turns:  10853 tokens  ⚠️ APPROACHING LIMIT
 15 turns:  16280 tokens  ⚠️ APPROACHING LIMIT
 20 turns:  21707 tokens  ❌ EXCEEDS LIMIT
 25 turns:  27133 tokens  ❌ EXCEEDS LIMIT
 30 turns:  32560 tokens  ❌ EXCEEDS LIMIT

Buffer memory will fail around turn 15
Reason: Linear token growth with no message removal


#### Checkpoint 3: Implementing the Window Memory Trim Function

Create a utility function that keeps conversations manageable by automatically trimming older messages beyond a set limit.

A constant `WINDOW_SIZE` is set so the chat keeps only the last `5` exchanges (10 messages total). 

A function called `trim_to_window` is also defined, which takes two parameters: 
- `history` (the chat history object) 
- `k` (the number of exchanges to keep, defaulting to `WINDOW_SIZE`)

Inside the function:
1. Check if the total number of messages in the `history` exceeds the window limit. 
2. If it does, extract only the most recent messages within the limit and store them temporarily in a variable named `recent_messages` to preserve the conversations that should be retained.
3. Use the `.clear()` method to clear the entire history.
4. Create a nested loop to add back only the extracted `recent_messages`, maintaining the sliding window of conversation context.
     
**Note:** Don't forget to run the cell and save the notebook before selecting "Test Work". Open the "Jupyter Help" toggle at the top of the notebook for more details.

In [6]:
WINDOW_SIZE = 5  # Keep last 5 exchanges

def trim_to_window(history, k=WINDOW_SIZE):
    """Keep only the last k exchanges (k*2 messages)"""
    if len(history.messages) > k * 2:
        recent_messages = history.messages[-(k * 2):]
        history.clear()
        for msg in recent_messages:
            history.add_message(msg)

#### Checkpoint 4: Running a Multi-Turn Support Conversation With Window Memory

Run the same support conversation using window memory to see how it keeps only recent exchanges while discarding older ones.

1. Define a session configuration dictionary named `window_config` that specifies the `"session_id"` as `"support_window"` to create a unique session identifier that isolates the memory storage for this conversation.

2. A loop structure with `print()` statements is provided to run through the conversation and display each turn.

3. Inside the loop, invoke the `conversation_chain` with the current `turn` as `"input"` and the session configuration, storing the result in the `response` variable so each message is sent to the chain while maintaining session context.

4. After each invocation, call the `trim_to_window()` function with the session's history from the `store` and `WINDOW_SIZE` to automatically remove older messages and maintain the sliding window of recent conversations.
   
**Note:** Don't forget to run the cell and save the notebook before selecting "Test Work". Open the "Jupyter Help" toggle at the top of the notebook for more details.

In [7]:
# Run window conversation
window_config = {"configurable": {"session_id": "support_window"}}

print("Running conversation with window memory...\n")
for i, turn in enumerate(support_turns, 1):
    response = conversation_chain.invoke({"input": turn}, config=window_config)
    # Enforce the sliding window limit here
    trim_to_window(store["support_window"], k=WINDOW_SIZE)
    print(f"Turn {i}: {turn}")

Running conversation with window memory...

Turn 1: My laptop won't start.
Turn 2: I tried charging it overnight.
Turn 3: The power light comes on but screen is black.
Turn 4: I tried holding power button for 30 seconds.
Turn 5: Yes, I removed all external devices.
Turn 6: The laptop is 2 years old, a Dell XPS 15.
Turn 7: No recent updates that I know of.
Turn 8: I can hear the fan running.


#### Checkpoint 5: Analyzing Window Memory Usage

Check what's left in window memory after trimming to see how it maintains stable token usage throughout the conversation.

1. Retrieve the chat history for the `"support_window"` session and store the messages in a variable named `window_messages` to access the limited conversation history maintained by the sliding window.

2. Join all message contents into a single string using spaces and store it in a variable named `window_content` to create a single text representation that enables token estimation.

3. Estimate the token count by multiplying the number of words in `window_content` by `1.3` to get an approximate token count since tokens don't map one-to-one with words. Store the result in a variable named `window_tokens`.

The `print()` statements are provided to display the window memory analysis results.

**Note:** Don't forget to run the cell and save the notebook before selecting "Test Work". Open the "Jupyter Help" toggle at the top of the notebook for more details.

In [8]:
# Analyze window memory usage
window_messages = get_session_history("support_window").messages 
window_content = " ".join([m.content for m in window_messages])
window_tokens = len(window_content.split()) * 1.3 # rough estimate

print("=== WINDOW MEMORY ANALYSIS ===\n")
print(f"Total messages stored: {len(window_messages)}")
print(f"Estimated tokens: {window_tokens:.0f}")

print(f"\nObservation: Memory size remains constant after {WINDOW_SIZE} exchanges")

=== WINDOW MEMORY ANALYSIS ===

Total messages stored: 10
Estimated tokens: 6458

Observation: Memory size remains constant after 5 exchanges


#### Compare Both Approaches

Let's compare buffer memory and window memory using the same 8-turn conversation. 

Buffer memory shows messages and tokens growing steadily with each turn until it hits the model's token limit and fails. Window memory keeps the same number of messages and tokens throughout, no matter how long the conversation runs. 

We'll also calculate how many tokens we save by switching to window memory, shown as both a number and percentage. The key insight: window memory avoids overflow by keeping only recent context instead of the entire conversation history.

In [9]:
print("=== BUFFER vs WINDOW MEMORY ===\n")

print("Buffer Memory (after 8 turns):")
print(f"  Messages: {len(buffer_messages)}")
print(f"  Tokens: {buffer_tokens:.0f}")
print(f"  Growth: Linear (unbounded)")
print(f"  Fails at: ~{failure_turn:.0f} turns\n")

print(f"Window Memory (after 8 turns, k={WINDOW_SIZE}):")
print(f"  Messages: {len(window_messages)}")
print(f"  Tokens: {window_tokens:.0f}")
print(f"  Growth: None (constant)")
print(f"  Fails at: Never\n")

reduction = ((buffer_tokens - window_tokens) / buffer_tokens) * 100
print(f"Token reduction: {buffer_tokens - window_tokens:.0f} tokens ({reduction:.0f}%)")

=== BUFFER vs WINDOW MEMORY ===

Buffer Memory (after 8 turns):
  Messages: 16
  Tokens: 8683
  Growth: Linear (unbounded)
  Fails at: ~15 turns

Window Memory (after 8 turns, k=5):
  Messages: 10
  Tokens: 6458
  Growth: None (constant)
  Fails at: Never

Token reduction: 2224 tokens (26%)


#### Demonstrate Scalability

Let's test window memory at scale by projecting token usage across conversations from 2 turns up to 1000 turns. 

No matter how long the conversation runs, window memory uses the same number of tokens because it keeps only the last `k` exchanges (in our case, 5 exchanges or 10 messages). While buffer memory crashes after a few dozen turns, window memory handles 1000+ turns without any issues.

This allows us to build chatbots with unlimited conversation lengths. The trade-off is that window memory forgets older context and keeps only recent messages, but that's acceptable for most applications.

In [10]:
print("=== SCALABILITY TEST ===\n")

# Calculate tokens per turn (average from current window)
tokens_per_turn = window_tokens / WINDOW_SIZE
token_limit = 16385

print("Window memory projection (constant token usage):\n")
for turns in [2, 5, 10, 20, 50, 100, 500, 1000]:
    projected = window_tokens  # Window maintains constant size
    
    if projected > token_limit:
        status = "❌ EXCEEDS LIMIT"
    elif projected > 3500:
        status = "⚠️ APPROACHING LIMIT"
    else:
        status = "✓"
    
    print(f"  {turns:>4} turns: ~{projected:>6.0f} tokens {status}")

print(f"\nWindow memory maintains constant ~{window_tokens:.0f} tokens")
print(f"Maximum capacity: ~{(window_tokens/token_limit)*100:.1f}% of {token_limit} token limit")
print(f"Result: Unlimited conversation length without token overflow ✓")

=== SCALABILITY TEST ===

Window memory projection (constant token usage):

     2 turns: ~  6458 tokens ⚠️ APPROACHING LIMIT
     5 turns: ~  6458 tokens ⚠️ APPROACHING LIMIT
    10 turns: ~  6458 tokens ⚠️ APPROACHING LIMIT
    20 turns: ~  6458 tokens ⚠️ APPROACHING LIMIT
    50 turns: ~  6458 tokens ⚠️ APPROACHING LIMIT
   100 turns: ~  6458 tokens ⚠️ APPROACHING LIMIT
   500 turns: ~  6458 tokens ⚠️ APPROACHING LIMIT
  1000 turns: ~  6458 tokens ⚠️ APPROACHING LIMIT

Window memory maintains constant ~6458 tokens
Maximum capacity: ~39.4% of 16385 token limit
Result: Unlimited conversation length without token overflow ✓


<details> 
<summary><strong>Note</strong></summary>

Observe that Buffer Memory stores every message permanently, causing token usage to grow linearly with conversation length. 

By running the conversation and inspecting memory, calculating tokens to measure current usage, and projecting growth to predict the failure point, we can diagnose that Buffer Memory will exceed token limits in long conversations — failing around turn 12 for GPT-3.5-turbo's 16385 token limit. 

The solution is to implement Window Memory, which keeps only the last `k` exchanges and maintains constant token usage, enabling unlimited conversation length. 

After fixing and verifying, we confirm that Window Memory successfully prevents token limit errors, though it trades complete history for efficiency by losing older conversation context.

</details>